In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, Blip2ForImageTextRetrieval

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Blip2ForImageTextRetrieval.from_pretrained("Salesforce/blip2-itm-vit-g-coco")
processor = AutoProcessor.from_pretrained("Salesforce/blip2-itm-vit-g-coco")
model = model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:

from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
train_ds = load_dataset("zai-org/VisionRewardDB-Image", split='train[:40000]')
test_ds = load_dataset("zai-org/VisionRewardDB-Image", split='train[40000:]')

import io, math, random
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd

df = pd.read_csv("rules.csv")

import pandas as pd
import re

df.columns = df.columns.str.strip()
df['Dimension'] = df['Dimension'].ffill()

df['dim_key'] = df['Dimension'].apply(lambda x: re.search(r'\((.*?)\)', x).group(1) if re.search(r'\((.*?)\)', x) else x)

guide = {
    dim_key: {
        int(row['Score']): row['Option'] + ": " +str(row['Description']).strip()
        for _, row in group.iterrows()
    }
    for dim_key, group in df.groupby('dim_key')
}


Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
dim_min = {i:min(guide[i].keys()) for i in guide.keys()}
dim_min

{'background': -1,
 'body': -4,
 'clarity': -2,
 'color aesthetic': -1,
 'color brightness': -1,
 'detail realism': -3,
 'detail refinement': -4,
 'emotion': -2,
 'face': -3,
 'hands': -4,
 'lighting aesthetic': -1,
 'lighting distinction': -1,
 'main object': -1,
 'object pairing': -1,
 'richness': -2,
 'safety': -3,
 'symmetry': -1,
 'unsafe type': 0}

In [ ]:
dims = {k: v for k, v in guide.items() if k not in ["unsafe type", "hands", "face", "body", "safety", "lighting aesthetic"]}.keys()
dims = list(dims)
dims

['background',
 'clarity',
 'color aesthetic',
 'color brightness',
 'detail realism',
 'detail refinement',
 'emotion',
 'lighting distinction',
 'main object',
 'object pairing',
 'richness',
 'symmetry']

In [ ]:
dim = random.choice(dims)
def format_data(sample):
    images = []
    messages = []
    dim = random.choice(dims)
    prompts = guide[dim]
    background_dict = guide[dim]
    sorted_items = sorted(background_dict.items())
    prompts = [f"About the {dim}: {value}" for key, value in sorted_items]
    inputs = processor(images=sample['image'], text=prompts, return_tensors="pt", padding=True)
    inputs["dim"] = [dims.index(dim)] * len(sample["image"])
    inputs["labels"] = torch.tensor([i[dim] for i in sample["annotation"]]) - dim_min[dim]
    return inputs

In [ ]:
train_ds = train_ds.with_transform(format_data)
test_ds = test_ds.with_transform(format_data)

In [ ]:
import wandb

In [ ]:
import torch.nn.functional as F
def focal_loss(logits, labels, gamma=2.0):
    # Calculate standard cross-entropy loss first.
    ce_loss = F.cross_entropy(logits, labels, reduction='none', label_smoothing=0.05)

    # Get softmax probabilities.
    pt = torch.exp(-ce_loss)

    # Compute focal loss.
    focal_loss = (1 - pt) ** gamma * ce_loss
    return focal_loss.mean()

In [ ]:
model

PeftModel(
  (base_model): LoraModel(
    (model): Blip2ForImageTextRetrieval(
      (vision_model): Blip2VisionModel(
        (embeddings): Blip2VisionEmbeddings(
          (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
        )
        (encoder): Blip2Encoder(
          (layers): ModuleList(
            (0-38): 39 x Blip2EncoderLayer(
              (self_attn): Blip2Attention(
                (qkv): Linear(in_features=1408, out_features=4224, bias=True)
                (projection): Linear(in_features=1408, out_features=1408, bias=True)
              )
              (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
              (mlp): Blip2MLP(
                (activation_fn): GELUActivation()
                (fc1): Linear(in_features=1408, out_features=6144, bias=True)
                (fc2): Linear(in_features=6144, out_features=1408, bias=True)
              )
              (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_aff

In [ ]:
import torch
from transformers import PreTrainedModel, PretrainedConfig

class Rater(PreTrainedModel):
    def __init__(self, backbone):
      super().__init__(PretrainedConfig())
      self.backbone = backbone
      # self.projections = torch.nn.ModuleList(
      #     [torch.nn.Linear(256, 256) for i in range(len(guide))]
      # )
      # for i in range(len(guide)):
      #   self.projections[i].weight.data.copy_(torch.eye(256))
      #   self.projections[i].bias = torch.nn.Parameter(torch.zeros(256))
      # self.projections = self.projections.to(device)
      self.t_score = torch.nn.Parameter(torch.ones(1))
      self.t_ce = torch.nn.Parameter(torch.ones(1))

    def forward(self, pixel_values, input_ids, attention_mask, dim, labels=None):
      dim = dim[0]
      outputs = self.backbone(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, use_image_text_matching_head=False)
      image_embeds = outputs.image_embeds
      text_embeds = outputs.text_embeds
      # image_embeds = self.projections[dim](image_embeds)
      image_embeds = image_embeds
      image_embeds = image_embeds
      image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
      text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
      logits_per_image_query_token = torch.matmul(image_embeds, text_embeds.t())
      logits_per_image = logits_per_image_query_token.max(dim=1)[0]
      outputs["logits_per_image"] = logits_per_image

      outputs.score = (torch.nn.functional.softmax(logits_per_image / self.t_score, dim=1) * (torch.arange(input_ids.shape[0])).to(self.device)).sum(1) #sum not mean

      if labels is not None:
        ce_loss = focal_loss(logits_per_image / self.t_ce, labels)
        # ce_loss = torch.nn.functional.cross_entropy(logits_per_image / self.t_ce, labels, label_smoothing=0.05)
        # gt_matrix = torch.nn.functional.one_hot(labels, num_classes=logits_per_image.shape[1]).to(device).float()
        # gt_matrix += 1e-3
        # gt_matrix = gt_matrix / (gt_matrix.sum(dim=0, keepdim=True))
        # gt_matrix = gt_matrix.detach()
        # print("logits_per_image.shape", logits_per_image.shape, "gt_matrix.shape", gt_matrix.shape)
        # print(gt_matrix)
        # assert logits_per_image.T.shape[-1] == pixel_values.shape[0]
        # assert torch.all(gt_matrix.T.sum(-1) >= 0.9), (gt_matrix.sum(-1), gt_matrix)
        # kl_loss = torch.nn.functional.cross_entropy(logits_per_image.T / self.t_kl, gt_matrix.T)
        mse_loss = torch.nn.functional.mse_loss(outputs.score, labels.float())
        rmse_loss = torch.sqrt(mse_loss)
        # loss = (ce_loss + kl_loss) / 2
        loss = (ce_loss + rmse_loss * 2) / 2
        # loss = (ce_loss + rmse_loss * 4 + kl_loss) / 3
        wandb.log({"ce_loss": ce_loss.item(), "rmse_loss": rmse_loss.item(), "loss": loss.item(), "acc": (logits_per_image.argmax(dim=1) == labels).float().mean().item()})
        outputs['loss'] = loss

      return outputs

In [ ]:
target_modules = ["model.vision_projection", "model.text_projection"]
for n, m in model.qformer.named_modules():
  if isinstance(m, torch.nn.modules.linear.Linear):
    target_modules.append(n)

In [ ]:
# torch.nn.functional.kl_div(torch.log(torch.tensor([0 + 1e-3, 1])), torch.nn.functional.log_softmax(torch.tensor([0, 1.0])), reduction='batchmean', log_target=True)

In [ ]:
# torch.nn.functional.kl_div(torch.nn.functional.log_softmax(torch.tensor([0, 1.0])), torch.tensor([0.0, 1]), reduction='batchmean')

In [ ]:
from peft import LoraConfig, TaskType
from peft import get_peft_model
config = LoraConfig(
    r=64, lora_alpha=64, target_modules=target_modules, modules_to_save=["model.query_tokens"], lora_dropout=0.05, bias="none"
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 19,365,888 || all params: 1,191,989,506 || trainable%: 1.6247


In [ ]:
my_rater = Rater(model).cuda()

In [ ]:
# my_rater = my_rater
# inputs = train_ds[:2].to("cuda")
# with torch.no_grad():
#   itm_score = my_rater(**inputs)
# # logits_per_image = itc_out.logits_per_image
# # probs = logits_per_image.softmax(dim=1)
# # print(itc_out.loss)
# # for i in range(len(texts)):
# #   print(f"{probs[0][i]:.1%} that image 0 is '{texts[i]}'")
# torch.nn.functional.softmax(itm_score.logits_per_image / 0.1, dim=-1)

In [ ]:
# (torch.nn.functional.softmax(itm_score.logits_per_image / 0.1, dim=1)*100).int()

In [ ]:
# ((torch.nn.functional.softmax(itm_score.logits_per_image / 0.1, dim=1)) * (torch.arange(inputs.input_ids.shape[0]).cuda())).sum(1)

In [ ]:
my_rater = my_rater.cuda()

In [ ]:
from transformers import TrainingArguments
import os
training_args = TrainingArguments(
    output_dir="BLIP2-Reward",
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=50,
    save_steps=1000,
    logging_steps=1,
    load_best_model_at_end=True,
    push_to_hub=True,
    remove_unused_columns=False,
    dataloader_num_workers=os.cpu_count(),
    fp16=True,
    lr_scheduler_kwargs={"num_cycles": 3},
    warmup_ratio=0.1,
    lr_scheduler_type="cosine_with_restarts"
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=my_rater,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=processor,
)

trainer.train()

In [ ]:
my_rater = my_rater
inputs = test_ds[:10].to("cuda")
with torch.no_grad():
  output = my_rater(**inputs)
(torch.nn.functional.softmax(output.logits_per_image / my_rater.t_score, dim=1) * (torch.arange(inputs.input_ids.shape[0])).to("cuda")).sum(1)

tensor([1.6221, 1.6156, 1.6232, 1.5704, 1.6027, 1.6165, 1.5083, 1.5872, 1.3450,
        1.6188], device='cuda:0', grad_fn=<SumBackward1>)

In [ ]:
my_rater.t_score

Parameter containing:
tensor([0.9776], device='cuda:0', requires_grad=True)